# BiLSTM Fact-Checking Classifier — All Conditions

Replicates and extends Alhindi et al. (2018) with synthetic justifications.

**Execution order:**
1. Cell 1 — Install + GloVe (once per session)
2. Cell 2 — Preprocessing (once per session)
3. Cell 3 — Model architecture and training functions (once per session)
4. Cells 4–7 — One per condition, each independent and self-saving
5. Cell 8 — Summary table

**Conditions:**
- C1: OpenAI justifications on Gemini-paraphrased claims
- C2: Gemini justifications on OpenAI-paraphrased claims
- C3: OpenAI justifications on original claims
- C4: Gemini justifications on original claims

## **Library**

In [ ]:
# CELL 1 — Install dependencies + download GloVe embeddings
# Run once at the start of each Colab session.
# GloVe 6B 100d: ~800MB — needed to initialise the embedding layer.

!pip install tensorflow pandas numpy scikit-learn openpyxl -q

import os
if not os.path.exists('glove.6B.100d.txt'):
    print('Downloading GloVe 100d (~800MB)...')
    !wget -q http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip glove.6B.100d.txt
    print('Done.')
else:
    print('GloVe already present.')

print('\n✅ Cell 1 complete')

Done.

✅ Cell 1 complete


## **PREPROCESSING**

In [ ]:
# CELL 2 — Preprocessing
#
# Covers all steps described in Section 4.6 of the thesis:
#   1. Binary label mapping (same grouping as Alhindi et al. 2018)
#   2. Train/val/test split reconstruction from 'split' column
#   3. Metadata extraction and normalisation (StandardScaler,
#      fitted on train only to prevent leakage)
#   4. GloVe loading and shared vocabulary construction
#   5. GloVe embedding matrix (maps vocab indices to 100d vectors)

import numpy as np
import pandas as pd
import json
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings
warnings.filterwarnings('ignore')

import random
# Fix random seeds for reproducibility across all runs
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)  # covers tf, numpy and python in one call

# ── GPU check ─────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs available: {len(gpus)}')
for g in gpus:
    print(f'  {g}')
if not gpus:
    print('WARNING: No GPU detected — training will be significantly slower.')

# ── 1. Load merged dataset ────────────────────────────────────────────────────
df = pd.read_excel('liar_plus_merged.xlsx')
df.columns = [c.strip() for c in df.columns]
print(f'\nDataset loaded: {len(df)} rows, {len(df.columns)} columns')

# ── 2. Binary label mapping ───────────────────────────────────────────────────
# Following Alhindi et al. (2018) Table 2:
#   FALSE = pants-fire, false, barely-true  →  label 0
#   TRUE  = half-true, mostly-true, true    →  label 1
FALSE_LABELS = {'false', 'barely-true', 'pants-fire'}
df['binary_label'] = df['label'].apply(lambda x: 0 if x in FALSE_LABELS else 1)
print(f'\nBinary label distribution:')
print(df['binary_label'].value_counts().to_string())

# ── 3. Reconstruct splits ─────────────────────────────────────────────────────
# The 'split' column was preserved during dataset creation (merge_dataset.ipynb).
# Using the original splits ensures direct comparability with published baselines.
train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'\nSplits: Train={len(train_df)} | Val={len(val_df)} | Test={len(test_df)}')

# ── 4. Metadata: numerical credit history features ────────────────────────────
# Following Alhindi et al. (2018) S+MJ condition:
# 5 credit history counts (past statements per veracity label) + their sum.
# Categorical fields (party, state, job_title) are excluded due to high missing rates in this dataset (up to 28% for job_title).
META_COLS = [
    'barely_true_count', 'false_count', 'half_true_count',
    'mostly_true_count', 'pants_fire_count'
]

def get_metadata(df_split):
    """Return 5 credit history counts + total as 6-dim float array."""
    meta   = df_split[META_COLS].fillna(0).values.astype(np.float32)   # nan = 0
    totals = meta.sum(axis=1, keepdims=True)
    return np.hstack([meta, totals])  # shape: (n_samples, 6)

# StandardScaler: zero mean, unit variance — fitted on train only
scaler     = StandardScaler()
meta_train = scaler.fit_transform(get_metadata(train_df))
meta_val   = scaler.transform(get_metadata(val_df))
meta_test  = scaler.transform(get_metadata(test_df))
print(f'\nMetadata shape (train): {meta_train.shape}  [5 credit counts + total, normalised]')

# ── 5. Shared tokeniser ───────────────────────────────────────────────────────
# Fitted on all text columns present in training across all conditions.
# Single shared vocabulary ensures consistent token indices across C1–C4.
# OOV token handles words seen at inference but absent from training vocabulary.
all_train_texts = (
    train_df['claim'].tolist() +
    train_df['openai_just_on_original'].tolist() +
    train_df['openai_just_on_paraphrased'].tolist() +
    train_df['gemini_just_on_original'].tolist() +
    train_df['gemini_just_on_paraphrased'].tolist() +
    train_df['openai_paraphrase'].tolist() +
    train_df['gemini_paraphrase'].tolist()
)
tokenizer = Tokenizer(oov_token='<OOV>')
tokenizer.fit_on_texts([str(t) for t in all_train_texts])
VOCAB_SIZE = len(tokenizer.word_index) + 1
print(f'Vocabulary size: {VOCAB_SIZE:,} tokens')

# ── 6. GloVe word embeddings ──────────────────────────────────────────────────
# GloVe 100d trained on 6B tokens (Pennington et al. 2014).
# Kept non-trainable throughout to limit overfitting on moderate data size,
# following Alhindi et al. (2018).
EMBED_DIM = 100

def load_glove(path='glove.6B.100d.txt'):
    """
    Load only GloVe vectors for words present in the tokeniser vocabulary.
    Filtering reduces RAM usage from ~800MB to a fraction of that,
    depending on vocabulary size — typically 10-30MB for LIAR-PLUS.
    """
    needed_words = set(tokenizer.word_index.keys())
    glove = {}
    with open(path, encoding='utf-8') as f:
        for line in f:
            word = line.split()[0]
            if word in needed_words:
                parts = line.split()
                glove[word] = np.array(parts[1:], dtype='float32')
    print(f'GloVe loaded: {len(glove):,} vectors (filtered from 400k)')
    return glove

glove = load_glove()


# ── 7. GloVe embedding matrix ─────────────────────────────────────────────────
# Maps each token index to its 100d GloVe vector.
# Tokens absent from GloVe receive a zero vector — they contribute o directional signal to the BiLSTM and are treated as uninformative.

def build_embedding_matrix(tokenizer, glove, vocab_size, embed_dim):
    matrix = np.zeros((vocab_size, embed_dim))
    hits, misses = 0, 0
    for word, idx in tokenizer.word_index.items():
        if idx < vocab_size:
            vec = glove.get(word)
            if vec is not None:
                matrix[idx] = vec
                hits += 1
            else:
                misses += 1
    print(f'Embedding matrix: {hits:,} hits | {misses:,} misses ({misses/vocab_size*100:.1f}% zero-init)')
    return matrix

embedding_matrix = build_embedding_matrix(tokenizer, glove, VOCAB_SIZE, EMBED_DIM)

# ── Global constants shared across all training cells ─────────────────────────
META_DIM = meta_train.shape[1]  # 6
print(f'Metadata dimension  : {META_DIM}')
print('\n✅ Cell 2 — Preprocessing complete')

GPUs available: 1
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

Dataset loaded: 12791 rows, 24 columns

Binary label distribution:
binary_label
1    7134
0    5657

Splits: Train=10240 | Val=1284 | Test=1267

Metadata shape (train): (10240, 6)  [5 credit counts + total, normalised]
Vocabulary size: 27,218 tokens
GloVe loaded: 20,503 vectors (filtered from 400k)
Embedding matrix: 20,503 hits | 6,714 misses (24.7% zero-init)
Metadata dimension  : 6

✅ Cell 2 — Preprocessing complete


## **MODEL**

In [ ]:
# CELL 3 — Model architecture and training functions
#
# Implements the BiLSTM architecture from Alhindi et al. (2018),
# S+MJ single-BiLSTM condition:
#
#   Text branch:
#     Input -> GloVe Embedding (100d, frozen)
#           -> BiLSTM (32 units x2 = 64-dim output)
#           -> Dropout (0.3)
#   Metadata branch:
#     Input (6-dim normalised credit history vector)
#   Merge:
#     Concatenate([text_out, meta]) -> Dense(1, sigmoid)
#
# Training config following the paper:
#   Optimiser: Adam, lr=1e-3
#   Loss: binary cross-entropy
#   Max epochs: 10 (+ early stopping on val_loss)
#   Batch size: 32
#
# run_condition() is the single entry point for each training cell.
# It encodes text, trains the model, evaluates on val/test,
# and saves results to Excel before returning.

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, Concatenate
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


def make_input_text(df_split, claim_col, just_col):
    """
    Concatenate claim and justification with a single space.
    No special separator token is used: the Keras tokeniser treats
    any string marker as a regular word, and a zero-vector OOV token
    in the middle of the sequence would introduce noise without adding
    structural information. Direct concatenation is consistent with
    Alhindi et al. (2018), who pass claim and justification as a single
    text sequence to the BiLSTM without explicit boundary markers.
    Missing values are filled with empty string to avoid NaN propagation.
    """
    claims = df_split[claim_col].fillna('').astype(str)
    justs  = df_split[just_col].fillna('').astype(str)
    return (claims + ' ' + justs).tolist()


# ── Dynamic MAX_LEN: set to max p95 across all conditions ────────────────────
# Ensures no condition is disproportionately truncated relative to others.
# A fixed MAX_LEN could systematically disadvantage models that generate
# longer justifications, confounding the cross-condition comparison.
len_stats = {}
for label, claim_col, just_col in [
    ('C1', 'gemini_paraphrase',  'openai_just_on_paraphrased'),
    ('C2', 'openai_paraphrase',  'gemini_just_on_paraphrased'),
    ('C3', 'claim',              'openai_just_on_original'),
    ('C4', 'claim',              'gemini_just_on_original'),
]:
    texts = make_input_text(train_df, claim_col, just_col)
    lens  = [len(t.split()) for t in texts]
    p95   = int(np.percentile(lens, 95))
    len_stats[label] = p95
    print(f'{label}: median={int(np.median(lens))} | p95={p95} | max={max(lens)}')

MAX_LEN = max(len_stats.values())
print(f'\nMAX_LEN set to: {MAX_LEN} (max p95 across all conditions)')



# ── Text encoding helpers ─────────────────────────────────────────────────────

def encode_texts(texts, max_len=MAX_LEN):
    """
    Tokenise and pad a list of strings to fixed length.
    Uses the shared tokeniser fitted in Cell 2.
    Sequences longer than max_len are truncated at the end;
    shorter ones are zero-padded at the end.
    """
    seqs = tokenizer.texts_to_sequences([str(t) for t in texts])
    return pad_sequences(seqs, maxlen=max_len, padding='post', truncating='post')

# ── Model builder ─────────────────────────────────────────────────────────────



def build_bilstm_model(lstm_units=32, dropout_rate=0.3):
    """
    Builds the BiLSTM model following Alhindi et al. (2018).

    Text branch
    -----------
    Embedding layer: GloVe 100d, frozen (trainable=False).
    Keeping embeddings frozen prevents overfitting on the
    ~10k training instances and preserves the generalisation
    captured during GloVe pre-training.

    BiLSTM layer: 32 units per direction (64 total).
    Bidirectional processing captures both left-to-right and
    right-to-left contextual dependencies within the claim-justification
    concatenation — important for political text where sentence-final
    hedges or negations can reverse the meaning of earlier content.

    Dropout: applied to the LSTM output to reduce overfitting.

    Metadata branch
    ---------------
    6-dim normalised credit history vector (5 label counts + total).
    Concatenated directly to the LSTM output before the final classifier,
    as in Alhindi et al. (2018) S+M and S+MJ conditions.

    Output
    ------
    Dense(1, sigmoid): binary probability (0=false, 1=true).
    Threshold at 0.5 for final label assignment.
    """
    # ── Text branch ───────────────────────────────────────────────────────────
    text_input = Input(shape=(MAX_LEN,), name='text_input')

    embed = Embedding(
        input_dim  = VOCAB_SIZE,
        output_dim = EMBED_DIM,
        weights    = [embedding_matrix],  # initialise with GloVe vectors
        trainable  = False,  # keep frozen throughout training
        mask_zero  = True,   # ignore padding tokens during LSTM computation
        name       = 'glove_embedding'
    )(text_input)

    lstm_out = Bidirectional(
        LSTM(lstm_units),  # 32 units per direction as in Alhindi et al. (2018)
        name='bilstm'
    )(embed)

    dropped = Dropout(dropout_rate, name='dropout')(lstm_out)

    # ── Metadata branch ───────────────────────────────────────────────────────
    meta_input = Input(shape=(META_DIM,), name='meta_input')

    # ── Merge + classify ──────────────────────────────────────────────────────
    # Concatenate text encoding (64-dim) and metadata (6-dim) -> 70-dim vector
    merged = Concatenate(name='merge')([dropped, meta_input])
    output = Dense(1, activation='sigmoid', name='output')(merged)

    model = Model(inputs=[text_input, meta_input], outputs=output)
    model.compile(
        optimizer = Adam(learning_rate=1e-3, clipnorm=1.0),  # as in Alhindi et al. (2018) + clipnorm for avoiding grad exploding
        loss      = 'binary_crossentropy',
        metrics   = ['accuracy']
    )
    return model

# ── Training pipeline ─────────────────────────────────────────────────────────

def run_condition(condition_name, claim_col, just_col,
                  output_file, epochs=30, batch_size=32, patience=5):
    """
    Full pipeline for one experimental condition.

    Steps:
      1. Encode: build claim + justification sequences and pad to MAX_LEN
      2. Class weights: computed from train labels to address imbalance
         (Hasan et al. 2025 show SMOTE yields no benefit on LIAR)
      3. Build a fresh BiLSTM model for this condition
      4. Train with early stopping on validation loss (patience=5)
      5. Evaluate on validation and test sets (accuracy, macro F1,
         weighted F1, confusion matrix)
      6. Save results to Excel immediately — safe against disconnections
      7. Save model artifacts for reproducibility (weights, tokeniser,
         scaler, config)

    Parameters
    ----------
    condition_name : str   label used in the results table
    claim_col      : str   column for claim text (original or paraphrased)
    just_col       : str   column for justification text
    output_file    : str   Excel file to save this condition's results
    """
    import pickle, json, os

    print(f'\n{"="*60}\nCondition: {condition_name}\n{"="*60}')
    print(f'  Claim column         : {claim_col}')
    print(f'  Justification column : {just_col}')

    # Step 1 — Encode text
    X_train = encode_texts(make_input_text(train_df, claim_col, just_col))
    X_val   = encode_texts(make_input_text(val_df,   claim_col, just_col))
    X_test  = encode_texts(make_input_text(test_df,  claim_col, just_col))

    y_train = train_df['binary_label'].values
    y_val   = val_df['binary_label'].values
    y_test  = test_df['binary_label'].values

    # Step 2 — Class weights
    # 'balanced' mode: weight_i = n_samples / (n_classes * count_i)
    classes           = np.unique(y_train)
    weights           = compute_class_weight('balanced', classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, weights))
    print(f'  Class weights        : {class_weight_dict}')

    # Step 3 — Fresh model (no weight sharing across conditions)
    model = build_bilstm_model()

    # Step 4 — Train with early stopping
    # Restores best weights (lowest val_loss) if training plateaus
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=patience,
        restore_best_weights=True,
        verbose=1
    )
    history = model.fit(
        [X_train, meta_train], y_train,
        validation_data = ([X_val, meta_val], y_val),
        epochs          = epochs,
        batch_size      = batch_size,
        class_weight    = class_weight_dict,
        callbacks       = [early_stop],
        verbose         = 1
    )

    # Step 5 — Evaluate
    # Accuracy is the primary metric for comparability with Alhindi et al. (2018).
    # Macro F1 is reported as secondary metric — unlike weighted F1, it treats
    # both classes equally and is more sensitive to minority class errors.
    # Confusion matrix is printed for qualitative inspection.
    def evaluate(X, meta, y_true, split_name):
        preds    = (model.predict([X, meta], verbose=0) > 0.5).astype(int).flatten()
        acc      = accuracy_score(y_true, preds)
        f1_w     = f1_score(y_true, preds, average='weighted')
        f1_macro = f1_score(y_true, preds, average='macro')
        cm       = confusion_matrix(y_true, preds)
        print(f'  [{split_name}] Accuracy: {acc:.4f} | Macro F1: {f1_macro:.4f} | Weighted F1: {f1_w:.4f}')
        print(f'  Confusion matrix:\n{cm}')
        print(f'  Classification report:\n{classification_report(y_true, preds, target_names=["false","true"])}')
        return acc, f1_w, f1_macro

    val_acc,  val_f1_w,  val_f1_macro  = evaluate(X_val,  meta_val,  y_val,  'VAL')
    test_acc, test_f1_w, test_f1_macro = evaluate(X_test, meta_test, y_test, 'TEST')

    # Step 6 — Save results immediately to Excel
    result = {
        'condition':    condition_name,
        'val_acc':      round(val_acc,      4),
        'val_f1_w':     round(val_f1_w,     4),
        'val_f1_macro': round(val_f1_macro,  4),
        'test_acc':     round(test_acc,     4),
        'test_f1_w':    round(test_f1_w,    4),
        'test_f1_macro':round(test_f1_macro, 4),
        'epochs_run':   len(history.history['loss'])
    }
    pd.DataFrame([result]).to_excel(output_file, index=False)
    print(f'\n  ✅ Results saved to {output_file}')

    # Step 7 — Save model artifacts for reproducibility
    # Each condition gets its own folder with weights, tokeniser, scaler, config.
    artifact_dir = output_file.replace('.xlsx', '_artifacts')
    os.makedirs(artifact_dir, exist_ok=True)

    # Model weights — can be reloaded with build_bilstm_model() + load_weights()
    model.save_weights(f'{artifact_dir}/model_weights.weights.h5')

    # Tokeniser — needed to encode new text at inference time
    with open(f'{artifact_dir}/tokenizer.pkl', 'wb') as f:
        pickle.dump(tokenizer, f)

    # Scaler — needed to normalise metadata at inference time
    with open(f'{artifact_dir}/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    # Config — all hyperparameters needed to reconstruct the model
    config = {
        'condition':   condition_name,
        'claim_col':   claim_col,
        'just_col':    just_col,
        'MAX_LEN':     MAX_LEN,
        'EMBED_DIM':   EMBED_DIM,
        'VOCAB_SIZE':  VOCAB_SIZE,
        'META_DIM':    META_DIM,
        'lstm_units':  32,
        'dropout':     0.3,
        'epochs_run':  len(history.history['loss'])
    }
    with open(f'{artifact_dir}/config.json', 'w') as f:
        json.dump(config, f, indent=2)

    print(f'  ✅ Artifacts saved to {artifact_dir}/')
    return result

print('✅ Cell 3 — Model functions ready')

C1: median=98 | p95=121 | max=194
C2: median=83 | p95=112 | max=196
C3: median=98 | p95=122 | max=588
C4: median=76 | p95=104 | max=541

MAX_LEN set to: 122 (max p95 across all conditions)
✅ Cell 3 — Model functions ready


## **1st Training** (OPENAI on paraphrased)

In [ ]:
# CELL 4 — C1: OpenAI justifications on Gemini-paraphrased claims
#
# Claim:         gemini_paraphrase  (Gemini-paraphrased version)
# Justification: openai_just_on_paraphrased  (OpenAI-generated)
#
# Mutual design: Gemini paraphrases the claim, OpenAI generates
# the justification on that paraphrase — no model sees its own
# paraphrase as justification input.
# Comparing C1 vs C3 isolates the effect of paraphrasing the claim
# on downstream classification with OpenAI justifications.

result_c1 = run_condition(
    condition_name = 'C1 — OpenAI just | Gemini paraphrase',
    claim_col      = 'gemini_paraphrase',
    just_col       = 'openai_just_on_paraphrased',
    output_file    = 'results_C1.xlsx'
)


Condition: C1 — OpenAI just | Gemini paraphrase
  Claim column         : gemini_paraphrase
  Justification column : openai_just_on_paraphrased
  Class weights        : {np.int64(0): np.float64(1.1408199643493762), np.int64(1): np.float64(0.8901251738525731)}
Epoch 1/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 8s 18ms/step - accuracy: 0.5726 - loss: 0.6726 - val_accuracy: 0.6145 - val_loss: 0.6473
Epoch 2/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.6108 - loss: 0.6518 - val_accuracy: 0.6301 - val_loss: 0.6424
Epoch 3/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6282 - loss: 0.6411 - val_accuracy: 0.6316 - val_loss: 0.6389
Epoch 4/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6376 - loss: 0.6335 - val_accuracy: 0.6340 - val_loss: 0.6363
Epoch 5/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6515 - loss: 0.6245 - val_accuracy: 0.6316 - val_loss: 0.6362
Epoch 6/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6614 - loss: 0.6174 - val_ac

## **2nd Training** (GEMINI on paraphrased)

In [ ]:
# CELL 5 — C2: Gemini justifications on OpenAI-paraphrased claims
#
# Claim:         openai_paraphrase  (OpenAI-paraphrased version)
# Justification: gemini_just_on_paraphrased  (Gemini-generated)
#
# Symmetric counterpart to C1: OpenAI paraphrases the claim,
# Gemini generates the justification on that paraphrase.
# Comparing C2 vs C4 isolates the paraphrasing effect for Gemini.
#

result_c2 = run_condition(
    condition_name = 'C2 — Gemini just | OpenAI paraphrase',
    claim_col      = 'openai_paraphrase',
    just_col       = 'gemini_just_on_paraphrased',
    output_file    = 'results_C2.xlsx'
)


Condition: C2 — Gemini just | OpenAI paraphrase
  Claim column         : openai_paraphrase
  Justification column : gemini_just_on_paraphrased
  Class weights        : {np.int64(0): np.float64(1.1408199643493762), np.int64(1): np.float64(0.8901251738525731)}
Epoch 1/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5638 - loss: 0.6765 - val_accuracy: 0.6207 - val_loss: 0.6498
Epoch 2/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6120 - loss: 0.6530 - val_accuracy: 0.6215 - val_loss: 0.6391
Epoch 3/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6249 - loss: 0.6437 - val_accuracy: 0.6332 - val_loss: 0.6337
Epoch 4/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6347 - loss: 0.6352 - val_accuracy: 0.6449 - val_loss: 0.6321
Epoch 5/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6466 - loss: 0.6262 - val_accuracy: 0.6417 - val_loss: 0.6321
Epoch 6/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6585 - loss: 0.6191 - val_ac

## **3rd Training** (OPENAI on original)

In [ ]:
# CELL 6 — C3: OpenAI justifications on original claims
#
# Claim:         claim  (original, unparaphrased)
# Justification: openai_just_on_original  (OpenAI-generated)
#
# No paraphrase applied — original claim text fed directly
# to the justification model and to the classifier.
# This is the primary synthetic condition for OpenAI:
# closest to the Alhindi et al. (2018) SJ setup, but with
# blind LLM-generated justifications instead of human ones.

result_c3 = run_condition(
    condition_name = 'C3 — OpenAI just | Original claim',
    claim_col      = 'claim',
    just_col       = 'openai_just_on_original',
    output_file    = 'results_C3.xlsx'
)


Condition: C3 — OpenAI just | Original claim
  Claim column         : claim
  Justification column : openai_just_on_original
  Class weights        : {np.int64(0): np.float64(1.1408199643493762), np.int64(1): np.float64(0.8901251738525731)}
Epoch 1/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5915 - loss: 0.6637 - val_accuracy: 0.6495 - val_loss: 0.6329
Epoch 2/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6206 - loss: 0.6468 - val_accuracy: 0.6449 - val_loss: 0.6278
Epoch 3/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6362 - loss: 0.6369 - val_accuracy: 0.6573 - val_loss: 0.6207
Epoch 4/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.6467 - loss: 0.6278 - val_accuracy: 0.6456 - val_loss: 0.6206
Epoch 5/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.6538 - loss: 0.6183 - val_accuracy: 0.6581 - val_loss: 0.6189
Epoch 6/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 10s 16ms/step - accuracy: 0.6728 - loss: 0.6082 - val_accuracy: 0.6456 - 

## **4th Training** (GEMINI on original)

In [ ]:
# CELL 7 — C4: Gemini justifications on original claims
#
# Claim:         claim  (original, unparaphrased)
# Justification: gemini_just_on_original  (Gemini-generated)
#
# Symmetric counterpart to C3 for Gemini.
# Primary synthetic condition for Gemini.

result_c4 = run_condition(
    condition_name = 'C4 — Gemini just | Original claim',
    claim_col      = 'claim',
    just_col       = 'gemini_just_on_original',
    output_file    = 'results_C4.xlsx'
)


Condition: C4 — Gemini just | Original claim
  Claim column         : claim
  Justification column : gemini_just_on_original
  Class weights        : {np.int64(0): np.float64(1.1408199643493762), np.int64(1): np.float64(0.8901251738525731)}
Epoch 1/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 7s 15ms/step - accuracy: 0.5686 - loss: 0.7083 - val_accuracy: 0.6308 - val_loss: 0.6492
Epoch 2/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6142 - loss: 0.6503 - val_accuracy: 0.6472 - val_loss: 0.6303
Epoch 3/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.6343 - loss: 0.6359 - val_accuracy: 0.6558 - val_loss: 0.6247
Epoch 4/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6498 - loss: 0.6264 - val_accuracy: 0.6488 - val_loss: 0.6247
Epoch 5/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6584 - loss: 0.6181 - val_accuracy: 0.6542 - val_loss: 0.6203
Epoch 6/30
320/320 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.6687 - loss: 0.6064 - val_accuracy: 0.6495 - v

## **SUMMARY**

In [ ]:
# CELL 8 — Summary table
#
# Loads individual result files (safe against partial runs)
# and combines them with published baselines:
#
#   Baseline A — Wang (2017), BiLSTM S condition
#     (claim only, no justification, no metadata)
#     Binary: val=0.59, test=0.60  [Table 2]
#
#   Baseline B — Alhindi et al. (2018), BiLSTM S+MJ condition
#     (claim + metadata + human justification)
#     Binary: val=0.71, test=0.68  [Table 2]
#
# F1 is not reported in binary setting in the original papers.
# Saved to results_summary.xlsx for thesis reporting.

import pandas as pd

baselines = [
    {
        'condition':  'Baseline A — Claim only [Wang 2017, BiLSTM S]',
        'val_acc':    0.59,  'val_f1':  None,
        'test_acc':   0.60,  'test_f1': None,
        'epochs_run': 'published'
    },
    {
        'condition':  'Baseline B — Human justification [Alhindi et al. 2018, BiLSTM S+MJ]',
        'val_acc':    0.71,  'val_f1':  None,
        'test_acc':   0.68,  'test_f1': None,
        'epochs_run': 'published'
    },
]

# Load individual condition files — prints warning if any are missing
condition_results = []
for fname in ['results_C1.xlsx', 'results_C2.xlsx', 'results_C3.xlsx', 'results_C4.xlsx']:
    try:
        condition_results += pd.read_excel(fname).to_dict('records')
        print(f'✅ Loaded: {fname}')
    except FileNotFoundError:
        print(f'⚠️  Missing: {fname} — run the corresponding training cell first')

all_results = baselines + condition_results
df_summary  = pd.DataFrame(all_results)
df_summary.to_excel('results_summary.xlsx', index=False)

print('\n' + '='*80)
print('COMPLETE RESULTS SUMMARY')
print('='*80)
print(df_summary[['condition', 'val_acc', 'test_acc', 'val_f1_w', 'val_f1_macro', 'test_f1_w', 'test_f1_macro', 'epochs_run']].to_string(index=False))
print('\n✅ Saved: results_summary.xlsx')

✅ Loaded: results_C1.xlsx
✅ Loaded: results_C2.xlsx
✅ Loaded: results_C3.xlsx
✅ Loaded: results_C4.xlsx

COMPLETE RESULTS SUMMARY
                                                          condition  val_acc  test_acc  val_f1_w  val_f1_macro  test_f1_w  test_f1_macro epochs_run
                      Baseline A — Claim only [Wang 2017, BiLSTM S]   0.5900    0.6000       NaN           NaN        NaN            NaN  published
Baseline B — Human justification [Alhindi et al. 2018, BiLSTM S+MJ]   0.7100    0.6800       NaN           NaN        NaN            NaN  published
                               C1 — OpenAI just | Gemini paraphrase   0.6238    0.6251    0.6232        0.6222     0.6249         0.6186         11
                               C2 — Gemini just | OpenAI paraphrase   0.6449    0.6148    0.6448        0.6442     0.6152         0.6093          9
                                  C3 — OpenAI just | Original claim   0.6581    0.6196    0.6582        0.6579     0.6199         